# Тәжірибелік жұмыс 1.8. Зерттеу моделін әзірлеу (нұсқа 3)

**Пән:** Педагогикалық зерттеулерді ұйымдастырудағы бұлтты технологиялар  
**Білім беру бағдарламасы:** 7M01511 Информатика  
**Орта:** Google Colab / Kaggle Notebooks (CPU жеткілікті)

## 1. Зерттеу сұрағы

**Зерттеу сұрағы:** Жобаның бастапқы кезеңіндегі көрсеткіштер бойынша қорытынды қорғау балын болжауға бола ма?

| Элемент | Сипаттама |
| --- | --- |
| Бақылау бірлігі | Бір магистранттың бір жобасы |
| X белгілері | `plan_score` (зерттеу жоспарының рубрика балы), `prototype_score` (алғашқы прототип балы), `ontime_share` (мерзімінде орындалған кезеңдер үлесі), `method_task_score` (бастапқы әдістемелік тапсырма нәтижесі) |
| y нәтижесі | `defense_score` — кейінгі қорытынды қорғаудың рубрика балы, 0–100 |
| Болжам жасалатын уақыт | Жобаның аралық кезеңі (қорғауға дейін). Барлық X деректері осы сәтке дейін алынған |
| Модель түрі | Регрессия: Ridge және RandomForestRegressor. Базалық: DummyRegressor |
| Метрикалар | MAE (негізгі), RMSE (қосымша), қателері ең үлкен үш жағдайды талдау |
| Педагогикалық қолдану | Аралық кезеңде әдістемелік кеңес беру бағыттарын анықтау |

**Шектеулер:**

- Қорытынды бағаның дайын құрамдастары X-ке көшірілмейді.
- Егер бастапқы балдар қорытынды формулаға кірсе, бұл байланыс ашық көрсетілуі тиіс. Мұнда жасанды деректің генерация ережесінде байланыс ашық жазылған (3-бөлім), ал қорытынды қорғау рубрикасының құрамдастары X-те жоқ.
- Рубрика нұсқасы сақталады: `rubric_version = R1` (X-те қолданылмайды, тек құжаттау үшін).
- Болжам баға қоюға негіз емес.

> Бұл жұмыста ЖИ моделі — деректермен оқытылған машиналық оқыту моделі. Дайын чатботтың жауабын алу модельді оқытуға жатпайды.

## 2. Кітапханалар, random seed және сақтау жолы

Сақтау жолы ортаға байланысты таңдалады: Kaggle-де `/kaggle/working`, басқа ортада (Colab, Deepnote, Binder, локальді) жоба ішіндегі `outputs` қалтасы.

In [1]:
import sys, platform, random
from pathlib import Path
from importlib import metadata

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Сақтау жолын ортаға бейімдеу
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)
print("Нәтиже сақталатын қалта:", OUT.resolve())
print("Python:", platform.python_version())
for pkg in ["numpy", "pandas", "scikit-learn", "joblib", "matplotlib"]:
    print(f"{pkg}: {metadata.version(pkg)}")

Нәтиже сақталатын қалта: /home/claude/work/proj/outputs
Python: 3.12.3
numpy: 2.4.4
pandas: 3.0.2
scikit-learn: 1.8.0
joblib: 1.5.3
matplotlib: 3.10.8


## 3. Деректерді жүктеу

Рұқсат етілген нақты деректер қолжетімсіз болғандықтан, **жасанды (synthetic) деректер** қолданылады. Бұл нақты оқу эксперименті емес, сондықтан нәтижеден нақты магистранттарға қатысты қорытынды жасалмайды. Байланыстар генерация ережесімен алдын ала берілген, сондықтан модель сапасы нақты жағдайдың көрсеткіші болмайды.

**Генерация ережесі** (300 тәуелсіз жоба, әр магистрантқа бір жоба):

- `plan_score` ~ Normal(65, 12), 0–100 аралығына шектеледі;
- `prototype_score` = 30 + 0.5·`plan_score` + Normal(0, 10), 0–100 аралығына шектеледі (жоспар мен прототип өзара байланысты);
- `ontime_share` ~ Beta(5, 2), яғни орташа шамамен 0,71;
- `method_task_score` ~ Normal(62, 13), 0–100 аралығына шектеледі;
- `defense_score = 5 + 0.30·plan_score + 0.28·prototype_score + 20·ontime_share + 0.15·method_task_score + ε`, мұндағы ε ~ Normal(0, 6). Нәтиже 0–100 аралығына шектеледі.

Яғни бастапқы балдар қорытынды балға **осы жасанды ережеде ашық түрде** әсер етеді. Нақты деректе мұндай байланыс болса, оны есепте міндетті түрде көрсету керек.

Сапаны тексеру үшін деректерге әдейі аздаған бос мән, ауқымнан тыс мән және қайталанған жолдар қосылады.

In [2]:
def make_synthetic(n=300, seed=SEED):
    rng = np.random.default_rng(seed)
    plan = np.clip(rng.normal(65, 12, n), 0, 100)
    proto = np.clip(30 + 0.5 * plan + rng.normal(0, 10, n), 0, 100)
    ontime = rng.beta(5, 2, n)
    method = np.clip(rng.normal(62, 13, n), 0, 100)
    noise = rng.normal(0, 6, n)
    y = 5 + 0.30 * plan + 0.28 * proto + 20 * ontime + 0.15 * method + noise
    return pd.DataFrame({
        "project_id": [f"P{i:03d}" for i in range(1, n + 1)],
        "plan_score": plan.round(1),
        "prototype_score": proto.round(1),
        "ontime_share": ontime.round(3),
        "method_task_score": method.round(1),
        "defense_score": np.clip(y, 0, 100).round(1),
        "rubric_version": "R1",
    })

def add_data_problems(df, seed=SEED):
    # Оқу мақсатында: бос мәндер, ауқымнан тыс мәндер, қайталанған жолдар
    rng = np.random.default_rng(seed + 1)
    df = df.copy()
    df.loc[rng.choice(df.index, 9, replace=False), "plan_score"] = np.nan
    df.loc[rng.choice(df.index, 8, replace=False), "method_task_score"] = np.nan
    df.loc[rng.choice(df.index, 6, replace=False), "ontime_share"] = np.nan
    df.loc[rng.choice(df.index, 3, replace=False), "ontime_share"] = 1.4       # қате енгізу
    df.loc[rng.choice(df.index, 2, replace=False), "prototype_score"] = 120    # қате енгізу
    dup = df.sample(4, random_state=seed)
    return pd.concat([df, dup], ignore_index=True)

DATA_PATH = OUT / "project_scores_synthetic.csv"
raw = add_data_problems(make_synthetic())
raw.to_csv(DATA_PATH, index=False)
raw = pd.read_csv(DATA_PATH)
print("Бастапқы кесте:", raw.shape)
raw.head()

Бастапқы кесте: (304, 7)


  project_id  plan_score  prototype_score  ontime_share  method_task_score  defense_score rubric_version
0       P001        68.7             81.6         0.600               56.7           73.6             R1
1       P002        52.5             40.9         0.545               63.2           50.9             R1
2       P003        74.0             75.6         0.742               37.0           65.7             R1
3       P004        76.3             64.9         0.728               68.2           79.2             R1
4       P005        41.6             50.2           NaN               62.0           51.9             R1

### Деректер сөздігі (data_dictionary)

In [3]:
data_dictionary = pd.DataFrame([
    ["project_id", "Жобаның жасырын идентификаторы (бір магистрант, бір жоба)", "мәтін", "Жоба басында", "P001–P300"],
    ["plan_score", "Зерттеу жоспарының рубрика балы", "балл", "Бастапқы кезең", "0–100"],
    ["prototype_score", "Алғашқы прототип балы", "балл", "Бастапқы кезең", "0–100"],
    ["ontime_share", "Мерзімінде орындалған кезеңдер үлесі", "үлес", "Аралық кезеңге дейін", "0–1"],
    ["method_task_score", "Бастапқы әдістемелік тапсырма нәтижесі", "балл", "Бастапқы кезең", "0–100"],
    ["defense_score", "Қорытынды қорғаудың рубрика балы (y)", "балл", "Қорғау кезінде", "0–100"],
    ["rubric_version", "Рубрика нұсқасы (X-те қолданылмайды)", "мәтін", "—", "R1"],
], columns=["баған", "мағынасы", "өлшем бірлігі", "қай уақытта алынды", "рұқсат етілген ауқым"])
data_dictionary.to_csv(OUT / "data_dictionary.csv", index=False)
data_dictionary

               баған                                           мағынасы өлшем бірлігі    қай уақытта алынды рұқсат етілген ауқым
0         project_id  Жобаның жасырын идентификаторы (бір магистрант...         мәтін          Жоба басында            P001–P300
1         plan_score                    Зерттеу жоспарының рубрика балы          балл        Бастапқы кезең                0–100
2    prototype_score                              Алғашқы прототип балы          балл        Бастапқы кезең                0–100
3       ontime_share               Мерзімінде орындалған кезеңдер үлесі          үлес  Аралық кезеңге дейін                  0–1
4  method_task_score             Бастапқы әдістемелік тапсырма нәтижесі          балл        Бастапқы кезең                0–100
5      defense_score               Қорытынды қорғаудың рубрика балы (y)          балл        Қорғау кезінде                0–100
6     rubric_version               Рубрика нұсқасы (X-те қолданылмайды)         мәтін            

## 4. Деректер сапасын тексеру

Тексеріледі: бос мәндер, қайталанған жолдар, ауқымнан тыс мәндер. Әр өзгерістің себебі `cleaning_log` тізіміне жазылады.

In [4]:
print("Бос мәндер:\n", raw.isna().sum(), "\n")
print("project_id бойынша қайталанған жолдар:", raw.duplicated("project_id").sum())

ranges = {
    "plan_score": (0, 100),
    "prototype_score": (0, 100),
    "ontime_share": (0, 1),
    "method_task_score": (0, 100),
    "defense_score": (0, 100),
}
for col, (lo, hi) in ranges.items():
    bad = ((raw[col] < lo) | (raw[col] > hi)).sum()
    print(f"{col}: ауқымнан тыс {bad} мән")

Бос мәндер:
 project_id           0
plan_score           9
prototype_score      0
ontime_share         6
method_task_score    9
defense_score        0
rubric_version       0
dtype: int64 

project_id бойынша қайталанған жолдар: 4
plan_score: ауқымнан тыс 0 мән
prototype_score: ауқымнан тыс 2 мән
ontime_share: ауқымнан тыс 3 мән
method_task_score: ауқымнан тыс 0 мән
defense_score: ауқымнан тыс 0 мән


In [5]:
cleaning_log = []
df = raw.copy()

# 1) Қайталанған жобаларды алып тастау
n0 = len(df)
df = df.drop_duplicates(subset="project_id", keep="first").reset_index(drop=True)
cleaning_log.append(f"Қайталанған жолдар алынды: {n0 - len(df)}. Себебі: бір жоба бір рет есептелуі керек.")

# 2) Ауқымнан тыс мәндерді NaN-ға ауыстыру (кейін Pipeline ішінде train медианасымен толтырылады)
for col, (lo, hi) in ranges.items():
    bad = (df[col] < lo) | (df[col] > hi)
    if bad.sum() > 0:
        df.loc[bad, col] = np.nan
        cleaning_log.append(f"{col}: ауқымнан тыс {int(bad.sum())} мән NaN-ға ауыстырылды. Себебі: қате енгізу.")

# 3) y бос жолдарды алып тастау (y толтырылмайды)
n1 = len(df)
df = df.dropna(subset=["defense_score"]).reset_index(drop=True)
cleaning_log.append(f"y бос жолдар алынды: {n1 - len(df)}. Себебі: нысана мәнін ойдан толтыруға болмайды.")

for line in cleaning_log:
    print("-", line)

print()
print("Тазартудан кейінгі жол саны:", len(df))
print("Тәуелсіз жобалар (магистранттар) саны:", df["project_id"].nunique())
print("Рубрика нұсқасы:", df["rubric_version"].unique())
print("Қалған бос мәндер (Pipeline ішінде train медианасымен толтырылады):")
print(df.isna().sum())

- Қайталанған жолдар алынды: 4. Себебі: бір жоба бір рет есептелуі керек.
- prototype_score: ауқымнан тыс 2 мән NaN-ға ауыстырылды. Себебі: қате енгізу.
- ontime_share: ауқымнан тыс 3 мән NaN-ға ауыстырылды. Себебі: қате енгізу.
- y бос жолдар алынды: 0. Себебі: нысана мәнін ойдан толтыруға болмайды.

Тазартудан кейінгі жол саны: 300
Тәуелсіз жобалар (магистранттар) саны: 300
Рубрика нұсқасы: <StringArray>
['R1']
Length: 1, dtype: str
Қалған бос мәндер (Pipeline ішінде train медианасымен толтырылады):
project_id           0
plan_score           9
prototype_score      2
ontime_share         9
method_task_score    8
defense_score        0
rubric_version       0
dtype: int64


In [6]:
df.describe().round(2)

       plan_score  prototype_score  ontime_share  method_task_score  defense_score
count      291.00           298.00        291.00             292.00         300.00
mean        64.49            62.18          0.71              60.27          65.14
std         11.09            11.76          0.17              12.83           9.32
min         34.20            20.60          0.17              22.40          40.40
25%         56.70            54.25          0.59              51.68          58.70
50%         64.40            61.85          0.72              61.95          65.40
75%         70.80            70.18          0.84              69.53          71.00
max         99.90            92.20          0.97              99.10          93.00

## 5. Train/test бөлу

Әр жол бір бөлек жобаны (бір магистрантты) сипаттайды, уақыт тізбегі жоқ және `project_id` бірегей. Сондықтан **80/20 кездейсоқ бөлу** қолданылады. Бір жоба train мен test-те қатар кездеспейді.

In [7]:
FEATURES = ["plan_score", "prototype_score", "ontime_share", "method_task_score"]
TARGET = "defense_score"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, df["project_id"], test_size=0.2, random_state=SEED
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train мен test-те ортақ project_id:", len(set(id_train) & set(id_test)))

Train: (240, 4)  Test: (60, 4)
Train мен test-те ортақ project_id: 0


## 6. Белгілерді дайындау

Бос мәндерді медианамен толтыру және масштабтау **тек train деректерінде үйретіледі**. Ол үшін барлық өңдеу `Pipeline` ішіне қойылады, сонда test деректерінің ақпараты оқытуға өтпейді (data leakage болмайды).

## 7. Базалық модель

`DummyRegressor(strategy="mean")` әр жобаға train бөлігінің орташа қорғау балын болжайды. Негізгі модель осыдан жақсы болмаса, қолданудың мәні жоқ.

In [8]:
def evaluate(model, X_, y_):
    pred = np.clip(model.predict(X_), 0, 100)       # болжамды 0–100 ауқымына шектеу
    mae = mean_absolute_error(y_, pred)
    rmse = np.sqrt(mean_squared_error(y_, pred))
    return mae, rmse, pred

baseline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", DummyRegressor(strategy="mean")),
]).fit(X_train, y_train)

mae_b, rmse_b, pred_b = evaluate(baseline, X_test, y_test)
print(f"Базалық модель (орташа): MAE = {mae_b:.2f}, RMSE = {rmse_b:.2f}")

Базалық модель (орташа): MAE = 8.38, RMSE = 10.05


## 8. Негізгі модельдер: Ridge және RandomForestRegressor

Әр модельде тек бір параметр өзгертіледі: Ridge үшін `alpha`, RandomForest үшін `max_depth`. Параметр **train ішіндегі 5-fold cross-validation** арқылы таңдалады. Негізгі модель де осы CV нәтижесімен таңдалады. Test бөлігі соңында бір-ақ рет қолданылады.

In [9]:
ridge_grid = GridSearchCV(
    Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", Ridge(random_state=SEED)),
    ]),
    param_grid={"model__alpha": [0.01, 0.1, 1, 10, 100]},
    scoring="neg_mean_absolute_error",
    cv=5,
).fit(X_train, y_train)

rf_grid = GridSearchCV(
    Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=200, random_state=SEED)),
    ]),
    param_grid={"model__max_depth": [2, 3, 4, 6]},
    scoring="neg_mean_absolute_error",
    cv=5,
).fit(X_train, y_train)

cv_ridge, cv_rf = -ridge_grid.best_score_, -rf_grid.best_score_
print(f"Ridge: alpha={ridge_grid.best_params_['model__alpha']}, CV MAE = {cv_ridge:.2f}")
print(f"RandomForest: max_depth={rf_grid.best_params_['model__max_depth']}, CV MAE = {cv_rf:.2f}")

if cv_ridge <= cv_rf:
    main_name, main_model = "Ridge", ridge_grid.best_estimator_
else:
    main_name, main_model = "RandomForest", rf_grid.best_estimator_
print("CV бойынша таңдалған негізгі модель:", main_name)
ridge, rf = ridge_grid.best_estimator_, rf_grid.best_estimator_

Ridge: alpha=0.01, CV MAE = 4.73
RandomForest: max_depth=6, CV MAE = 5.44
CV бойынша таңдалған негізгі модель: Ridge


## 9. Бағалау

Үш модель бір test бөлігінде салыстырылады.

- **MAE** — болжам мен нақты балл арасындағы абсолютті айырманың орташа мәні, балмен өлшенеді. Мысалы, MAE = 5 болса, болжам орта есеппен 5 баллға қателеседі.
- **RMSE** — қателерді квадраттап орташалайды да түбір алады. Үлкен қателерге сезімтал.

In [10]:
mae_r, rmse_r, pred_r = evaluate(ridge, X_test, y_test)
mae_f, rmse_f, pred_f = evaluate(rf, X_test, y_test)
mae_m, rmse_m, pred_m = (mae_r, rmse_r, pred_r) if main_name == "Ridge" else (mae_f, rmse_f, pred_f)

metrics = pd.DataFrame([
    ["DummyRegressor (орташа)", mae_b, rmse_b],
    [f"Ridge (alpha={ridge_grid.best_params_['model__alpha']})", mae_r, rmse_r],
    [f"RandomForest (max_depth={rf_grid.best_params_['model__max_depth']})", mae_f, rmse_f],
], columns=["модель", "MAE", "RMSE"]).round(2)

metrics["MAE жақсару, % (базалыққа қатысты)"] = ((mae_b - metrics["MAE"]) / mae_b * 100).round(1) + 0.0
metrics.to_csv(OUT / "metrics.csv", index=False)
metrics

                       модель   MAE   RMSE  MAE жақсару, % (базалыққа қатысты)
0     DummyRegressor (орташа)  8.38  10.05                                 0.1
1          Ridge (alpha=0.01)  5.10   6.80                                39.2
2  RandomForest (max_depth=6)  5.27   6.79                                37.1

In [11]:
print("Ridge коэффициенттері (стандартталған белгілер):")
print(pd.Series(ridge.named_steps["model"].coef_, index=FEATURES).round(2).sort_values(key=abs, ascending=False))
print()
print("RandomForest белгі маңыздылығы:")
print(pd.Series(rf.named_steps["model"].feature_importances_, index=FEATURES).round(3).sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(y_test, pred_m, alpha=0.7, label=main_name)
ax.plot([0, 100], [0, 100], "k--", linewidth=1, label="Идеал болжам")
ax.set_xlabel("Нақты қорғау балы")
ax.set_ylabel("Болжам")
ax.set_title("Нақты және болжам балл (test)")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "pred_vs_actual.png", dpi=150)
plt.show()

Ridge коэффициенттері (стандартталған белгілер):
ontime_share         3.53
plan_score           3.48
prototype_score      3.35
method_task_score    1.76
dtype: float64

RandomForest белгі маңыздылығы:
prototype_score      0.350
plan_score           0.298
ontime_share         0.228
method_task_score    0.125
dtype: float64


## 10. Қате талдауы: қателері ең үлкен үш жағдай

In [12]:
pred_table = pd.DataFrame({
    "project_id": id_test.values,
    "y_true": y_test.values,
    "y_pred_baseline": pred_b.round(1),
    "y_pred_ridge": pred_r.round(1),
    "y_pred_rf": pred_f.round(1),
    "y_pred_main": pred_m.round(1),
})
pred_table["abs_error_main"] = (pred_table["y_true"] - pred_table["y_pred_main"]).abs().round(1)
pred_table["signed_error_main"] = (pred_table["y_pred_main"] - pred_table["y_true"]).round(1)
pred_table = pd.concat([pred_table, X_test.reset_index(drop=True)], axis=1)
pred_table.to_csv(OUT / "predictions.csv", index=False)

worst = pred_table.sort_values("abs_error_main", ascending=False).head(3)
worst[["project_id", "y_true", "y_pred_main", "signed_error_main"] + FEATURES]

   project_id  y_true  y_pred_main  signed_error_main  plan_score  prototype_score  ontime_share  method_task_score
45       P064    79.0         62.2              -16.8        72.0             55.4         0.556               59.3
4        P234    89.5         73.6              -15.9        79.5             74.3         0.745               56.0
28       P158    58.4         73.1               14.7        76.8             59.9         0.910               64.1

**Қате туралы түсіндірме:** үш жағдайды `signed_error_main` бағанымен қараңыз (оң таңба — модель қорғау балын жоғары болжаған, теріс — төмен болжаған). Жасанды деректе мұндай қателердің негізгі себебі — генерация ережесіндегі кездейсоқ шу ε, яғни белгілері ұқсас жобалардың қорғау нәтижесі әртүрлі шығады. Нақты деректе қосымша себептер болуы мүмкін: жетекші өзгерісі, тақырып күрделілігі, қорғау кезіндегі жағдай, бағалаушылардың келіспеушілігі. Бұл ақпарат X-те жоқ.

**Педагогикалық салдар:** қорғау балы жоғары болжанып, іс жүзінде төмен шыққан жоба кеңессіз қалуы мүмкін. Сондықтан болжам тек әдістемелік кеңес бағытын таңдауға көмекші, оны баға қою үшін қолдануға болмайды.

## 11. Жаңа мысалдарға болжам

Үш жаңа (ойдан құрылған) жоба үшін болжам жасалады.

In [13]:
new_projects = pd.DataFrame([
    {"plan_score": 48, "prototype_score": 50, "ontime_share": 0.40, "method_task_score": 45},
    {"plan_score": 68, "prototype_score": 66, "ontime_share": 0.75, "method_task_score": 60},
    {"plan_score": 90, "prototype_score": 85, "ontime_share": 1.00, "method_task_score": 88},
])[FEATURES]

new_projects["болжам_қорғау_балы"] = np.clip(main_model.predict(new_projects), 0, 100).round(1)
new_projects

   plan_score  prototype_score  ontime_share  method_task_score  болжам_қорғау_балы
0          48               50          0.40                 45                47.7
1          68               66          0.75                 60                68.2
2          90               85          1.00                 88                90.1

## 12. Нәтижелерді сақтау

Модельмен бірге **белгілер реті**, **енгізу ауқымы**, **деректің жасанды екені** туралы белгі, **рубрика нұсқасы** және **кітапхана нұсқалары** сақталады. Веб-қосымша осы файлды ашып, жаңа деректі сол Pipeline арқылы өңдейді.

In [14]:
bundle = {
    "pipeline": main_model,
    "model_name": main_name,
    "features": FEATURES,
    "target": TARGET,
    "input_ranges": {
        "plan_score": [0, 100],
        "prototype_score": [0, 100],
        "ontime_share": [0, 1],
        "method_task_score": [0, 100],
    },
    "output_range": [0, 100],
    "rubric_version": "R1",
    "data_is_synthetic": True,
    "metrics": {"MAE_test": round(mae_m, 2), "MAE_baseline": round(mae_b, 2)},
    "versions": {
        "python": platform.python_version(),
        "scikit-learn": metadata.version("scikit-learn"),
        "pandas": metadata.version("pandas"),
        "numpy": metadata.version("numpy"),
    },
}
joblib.dump(bundle, OUT / "research_model.joblib")

# requirements.txt: GitHub / Streamlit Cloud үшін
reqs = ["numpy", "pandas", "scikit-learn", "joblib"]
lines = [f"{p}=={metadata.version(p)}" for p in reqs] + ["streamlit"]
(OUT / "requirements.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

# Қайта жүктеп тексеру
check = joblib.load(OUT / "research_model.joblib")
test_row = pd.DataFrame([[65, 65, 0.7, 60]], columns=check["features"])
print("Тексеру болжамы:", np.clip(check["pipeline"].predict(test_row), 0, 100).round(1))
print("Негізгі модель:", check["model_name"])
print("Сақталған файлдар:", sorted(p.name for p in OUT.iterdir()))
print("Нұсқалар:", check["versions"])

Тексеру болжамы: [65.9]
Негізгі модель: Ridge
Сақталған файлдар: ['data_dictionary.csv', 'metrics.csv', 'pred_vs_actual.png', 'predictions.csv', 'project_scores_synthetic.csv', 'requirements.txt', 'research_model.joblib']
Нұсқалар: {'python': '3.12.3', 'scikit-learn': '1.8.0', 'pandas': '3.0.2', 'numpy': '2.4.4'}


## 13. Қорытынды сұрақтарға жауап беруге арналған жазбалар

1. **Неге бұл модельді таңдадыңыз?** Таңдау train ішіндегі CV MAE бойынша жасалды. Ridge түсіндіруге оңай, ал RandomForest сызықтық емес байланысты ұстай алады.
2. **Базалық тәсілден артықшылығы бар ма?** `metrics` кестесіндегі MAE мәндерін салыстырыңыз. Дерек жасанды екенін ұмытпаңыз.
3. **Қандай қате маңызды?** Қорғау балы жоғары болжанғанымен төмен шыққан жоба, яғни кеңес қажет магистрант байқалмай қалады.
4. **Жаңа топқа қолданар алдында нені тексересіз?** Рубрика нұсқасының сәйкестігін, X белгілерінің мағынасын, жаңа топтағы MAE мәнін және кіші топтар бойынша қателерді.

> ЖИ көмекшісін қолдансаңыз, есепте қай кодты соның көмегімен жазғаныңызды және қалай тексергеніңізді көрсетіңіз.